<font color=white size=8 face=雅黑>**切表面fromDict**</font>

In [ ]:
from ModifyStructure import BulkToSlabGenerator

# str1 = Structure.from_file("/data2/home/luodh/999/CONTCAR")
config = {
    # 1. 初始化部分
    "structure_source": "/data2/home/luodh/888/",  # 也可以是 "path/to/POSCAR"
    "save_dir": "/data2/home/luodh/888/999", #保存的文件路径
    
    # 2. 生成参数部分
    "generate_params": {
        "miller_indices": "110", #切表面的index
        "target_layers": 20, #目标层数
        "hcluster_cutoff": 0.1, #读取原子层数的阈值，金属大一点，氧化物小一点
        "vacuum_thickness": 10.0, #真空层大小
        "supercell_matrix": "1x1x1", #超胞
        "fix_bottom_layers": 8 #固定层数
    },
    
    # 3. 保存选项
    "save_options": {
        "save": True, #是否保存
        "filename_prefix": "POSCAR" #文件的开头
    }
}
slabs = BulkToSlabGenerator.run_from_dict(config)




<font color=white size=8 face=雅黑>**添加吸附质的方法**</font>

In [ ]:
from ModifyStructure import AdsorptionModify
#用于测试分析参数
# config = {
#     "target_slab_source": "/data2/home/luodh/888/999/POSCAR_110_20L_term1.vasp", # 假设存在
#     "save_dir": "/data2/home/luodh/888/999/ads",
#     "mode": "analyze", #三种模式，分析确定位点analyze，获取吸附结构generate，依照相关结构获取吸附结构
#     "log_to_file": True,
    
#     "analyze_params": {
#         "describe_site_index": "all", # 查看 0 号位点详情
#     },
    
#     "plot": True,
#     "plot_params": {
#         "figsize": (8, 8),          # 画布大小 (会被 run_from_dict 提取使用)
#         "repeat": [1, 1, 1],        # 扩胞倍数 (用于显示更广的区域)
#         "scale": 1.0,               # 原子半径缩放
#         "window": 1.0,              # z方向窗口大小 (控制显示的层数)
#         "decay": 0.1,               # 透明度衰减
#         "label_offset": (0.1, 0.2), # 编号标签的偏移量 (x, y)
#         "inverse": False            # 是否反转颜色/背景
#     }
# }
# AdsorptionModify.run_from_dict(config)

# config2 = {
#     "target_slab_source": "/data2/home/luodh/999/1000/", #设置结构
#     "save_dir": "/data2/home/luodh/999/1000/ads", #吸附结构放的位置
#     "mode": "generate", #用于生成结构的方法
#     "log_to_file": True,
#     "generate_params": {
#         "molecule_formula": "CH3", #设置吸附分子
#         "find_args": {
#             "positions": ["ontop"], # 吸附位点的类型,目前支持这几种hollow，bridge，ontop
#             "distance": 1.8, #吸附分子的距离
#         }
#     },
#     "plot": True,
# }
# AdsorptionModify.run_from_dict(config2)

<font color=white size=8 face=雅黑>**设计掺杂结构的方法**</font>

In [ ]:
from ModifyStructure import StructureModify
from pymatgen.symmetry.analyzer import SpacegroupAnalyzer
from pymatgen.core import Structure
import os
str1 = Structure.from_file("/data2/home/luodh/888/999/POSCAR_110_20L_term0.vasp")
# ##生成掺杂结构的方法,尤其是对于合金来说

doped_structures = StructureModify.generate(
    structure=str1,
    substitute_element="Fe",  # 要被替换的元素
    dopant="Cr",               # 掺杂元素
    dopant_num=4,             # 每次替换 2 个原子
    num_structs=10,            # 生成 5 个不重复的结构
    random_seed=42            # 固定随机种子以便复现
)
print(f"成功生成了 {len(doped_structures)} 个掺杂结构。")
#在class外进行保存功能最合适
for i, s in enumerate(doped_structures):
    comp = s.composition
    filename = os.path.join("/data2/home/luodh/888/999/", f"POSCAR_{i}")
    s.to(filename=filename, fmt="poscar")
    print(f"结构 {i+1}: {comp.formula} (P 原子数: {comp['P']})")


# str1 = Structure.from_file("/data2/home/luodh/999/1000/POSCAR_110_5L.vasp")
# paths = StructureModify.step_by_step(
#     structure=str1,
#     substitute_element="Cu",
#     dopant="Fe",
#     max_steps=3,
#     max_structures_num=10,
#     return_all_steps=True
# )
# base_dir = "/data2/home/luodh/999/1000/"
# for path_idx, path in enumerate(paths):
#     path_dir = os.path.join(base_dir, f"path_{path_idx}")
#     os.makedirs(path_dir, exist_ok=True)
    
#     print(f"--- 路径 {path_idx} 分析 ---")
#     for step_idx, struct in enumerate(path):
#         # 保存
#         fname = os.path.join(path_dir, f"POSCAR_step_{step_idx}")
#         struct.to(filename=fname, fmt="poscar")
        
#         # 【新增】验证对称性
#         sga = SpacegroupAnalyzer(struct, symprec=0.01) # symprec 是精度，0.01 是标准值
#         sg_symbol = sga.get_space_group_symbol()
#         sg_number = sga.get_space_group_number()
        
#         print(f"  Step {step_idx}: 掺杂数={step_idx+1} | 空间群: {sg_symbol} ({sg_number})")

<font color=white size=8 face=雅黑>**写入Vasp计算文件的方法**</font>

In [ ]:
from VaspInput import VaspInputMaker

#----Bulk----
# config = {
#     "structure": "/data2/home/luodh/888", # 实际中这里可以是文件路径字符串
#     "functional": "BEEF",
#     "is_metal": True, #选择金属体系
#     "kpoints_density":50, #设置K点密度，以xyz的方法设置
#     "user_incar_settings": {"NPAR": 4}, #特殊的INCAR设置
#     "extra_kwargs": {"user_potcar_functional": None} #额外的设置
# }
# maker_from_dict = VaspInputMaker.from_dict_ecat(config)
# dict_dir = maker_from_dict.write_bulk("/data2/home/luodh/888/1")


# #----Slab----
# config = {
#     "structure": "/data2/home/luodh/888",
#     "functional": "BEEF",         # 自动开启 BEEF-vdW 参数
#     "auto_dipole": True,          # 自动计算偶极方向
    
#     # 强制指定 K 点 (覆盖默认的 density 逻辑)
#     "normal_kpoints_set": False,
#     "user_kpoints_settings": Kpoints.monkhorst_automatic(kpts=(4,4,1)), #特殊的k点设置
#     "user_incar_settings": {
#         "LVHAR": True,            # 输出静电势
#         "IDIPOL": 3               # 确保是 Z 方向 (虽然 auto_dipole 会尝试检测)
#     }
# }
# maker_from_dict = VaspInputMaker.from_dict_ecat(config)
# dict_dir = maker_from_dict.write_slab("/data2/home/luodh/888/1")


#---noscf---
# config_dos = {

#     # 关键：指定上一步计算的目录
#     "prev_dir": "/data2/home/luodh/spinel-slab/Al2O4_Spinel/MnAl2O4/slab_110_1",
    
#     # 静态计算参数
#     "functional": "PBE",
#     "number_of_docs": 2001,       # NEDOS
#     "kpoints_density": 25,        # 更密的 K 点
    
#     "user_incar_settings": {
#         "LOBRIT": 11,
#         "ISMEAR": -5              # 四面体方法
#     }
# }
# maker_from_dict = VaspInputMaker.from_dict_ecat(config_dos)
# dict_dir = maker_from_dict.write_noscf(output_dir="/data2/home/luodh/888/1")
# config_dos_direct = {
#     "structure": "/data2/home/luodh/888",
#     "functional": "PBE",
#     "number_of_docs": 2001,
#     "kpoints_density": 25,
#     "user_incar_settings": {
#         "LORBIT": 11,
#         "ISMEAR": -5
#     }
# }

# maker = VaspInputMaker.from_dict_ecat(config_dos_direct)
# out_dir = maker.write_noscf(output_dir="/data2/home/luodh/888/1")

#---Lobster---
config_lobster_prev = {
    "functional": "PBE",
    "kpoints_density": 60,
    "structure": "/data2/home/luodh/888/",
#    "prev_dir": "/data2/home/luodh/888",
    "user_incar_settings": {
        "LORBIT": 11,
        "ISMEAR": -5,
        "SIGMA": 0.20,
    },
    "user_supplied_basis": None,  # 或者填你自定义basis字典
    "overwritedict": {            # lobsterin 覆盖项（可选）
        "COHPstartEnergy": -25.0,
        "COHPendEnergy": 25.0,
        "cohpGenerator": "from 0.1 to 1.2 orbitalwise",
    }
}
maker = VaspInputMaker.from_dict_ecat(config_lobster_prev)
out_dir = maker.write_lobster(output_dir="/data2/home/luodh/888/1")

In [ ]:
from ModifyStructure import StructureModify
from pymatgen.core import Structure
from pathlib import Path
import os
from Vaspinput import VaspInputMaker
from pymatgen.io.vasp import Poscar

work_dir = Path("/data2/home/luodh/high-calc/structure").resolve()
input_file = work_dir / "POSCAR_PtSnCu"
stru1 = Structure.from_file(input_file)
user_incar_setting = {
    "MAGMOM":{'Co':5.0, 'Pt':3.0},
    "LDAUU":{'Pt':{'Sn':5.0}}
}
input = VaspInputMaker(structure=stru1, user_incar_settings=user_incar_setting).write_slab(output_dir="/data2/home/luodh/999/2")
# target_metals = ["Sc", "Ti", "V", "Cr", "Mn", "Fe", "Co", "Ni", "Zn"]
# for metal in target_metals:
#     modifier = StructureModify(stru1)
#     species_map = {"Cu": metal}
#     new_structure = modifier.replace_species_all(species_map)
#     output_filename = f"POSCAR_PtSn{metal}"
#     output_path = work_dir / output_filename
#     Poscar(new_structure).write_file(output_path)



In [3]:
from VaspInput import VaspInputMaker

maker = VaspInputMaker(
    functional="BEEF",                 # 这里不会强制覆盖 prev_dir 的 functional；freq 会从 prev INCAR 推断
    kpoints_density=50,               # 对 freq 通常不重要（因为会继承 KPOINTS）
    user_incar_settings={             # 这里是“最终覆盖”
        "EDIFF": 1e-7,
    },
)

maker.write_freq(
    output_dir="/data2/home/luodh/999/23",
    prev_dir="/data2/home/luodh/work/DICP/slab-Pd/PdCx/subC/reaction-6C/C2H2ads/clean-zone/one-atom",
    adsorbate_formula="C2H2", ##提供吸附的分子式，如果表面也存在和吸附质相同的元素，需要提供准确的吸附质indices
    adsorbate_formula_prefer="tail",
)


Picked vibrate indices by formula C8H2: [80, 81, 82, 83, 84, 85, 86, 87, 88, 89]


/home/user/.conda/envs/workflow/lib/python3.11/site-packages/pymatgen/io/vasp/sets.py:486: BadInputSetWarning: POTCAR data with symbol Pd is not known by pymatgen to correspond with the selected user_potcar_functional='PBE'. This POTCAR is known to correspond with functionals ['PBE_52', 'PBE_54']. Please verify that you are using the right POTCARs!
  potcar="\n".join(self.potcar_symbols) if potcar_spec else self.potcar,


'/data2/home/luodh/999/23'

In [ ]:
from Vaspanalysis import CohpAnalysis ## 从Vaspanalysis模块导入CohpAnalysis类，用于分析COHP数据
from pymatgen.electronic_structure.core import Spin ##导入自旋

#初始化类， work_dir 是cohpcar文件所在目录，cohpcar_file 是cohpcar文件名, 二者输入其一即可。 output_dir 是输出目录， 
cohp = CohpAnalysis(work_dir="/data2/home/luodh/cohp/cohp", output_dir = "/data2/home/luodh/cohp")
##获取键的ICOHP摘要信息, bond_label:键的序号，和ICOHP中序号一致，一定得是str类型
cohp.get_icohp_summary(bond_label="1")
# #获取键的成键轨道信息，bond_label: 键的序号，和ICOHP中序号一致，一定得是str类型
cohp.get_orbital_bond_info(bond_label="1")
##绘制总键COHP/ICOHP曲线, bond_label: 键的序号，和ICOHP中序号一致，一定得是str类型, data_type: 数据类型，COHP或ICOHP, spin: 自旋, xlim: x轴范围(能量范围)
cohp.plot_total_bond_cohp(bond_label="1", data_type="COHP", spin=Spin.up, xlim=(-10, 20))
##导出总键COHP/ICOHP数据到CSV文件, bond_label: 键的序号，和ICOHP中序号一致，一定得是str类型, data_type: 数据类型，COHP或ICOHP, spin: 自旋，output_filename: 输出文件名
cohp.save_total_bond_data_to_csv(bond_label="1", data_type="COHP", spin=Spin.down, output_filename="total_cohp.csv")
#绘制图片，bond_label为键的序号，orbital_label为轨道的标签，data_type为数据类型，spin为自旋，xlim为x轴范围
cohp.plot_orbital_cohp(bond_label="1", orbital_label="2px-3dx2", data_type="COHP", spin=Spin.down, xlim=(-20, 20))
#导出数据，bond_label为键的序号，orbital_label为轨道的标签，data_type为数据类型，spin为自旋，output_filename为输出文件名
cohp.save_orbital_data_to_csv(bond_label="1", orbital_label="6s-2py", data_type="COHP", spin=Spin.up)

In [11]:
from VaspInput import VaspInputMaker
from pymatgen.core import Structure
from pathlib import Path

work_dir = Path("/data2/home/luodh/999/CONTCAR").resolve()
out_dir = Path(work_dir).parent / "1010"
str1 = Structure.from_file(work_dir)

user_incar_settings = {"IBRION1": -2000, "LDAUU":{"Cu":-5}}
maker = VaspInputMaker(structure=str1,user_incar_settings=user_incar_settings).write_bulk(output_dir=out_dir)

<string>:31: BadInputSetWarning: Overriding the POTCAR functional is generally not recommended  as it significantly affects the results of calculations and compatibility with other calculations done with the same input set. Note that some POTCAR symbols specified in the configuration file may not be available in the selected functional.
